# PoreMind 面向对象逐步骤分析
需求对应六步：降噪→事件检测→特征提取→异常过滤→建模选优→新样本分类。

In [ ]:
from poremind import create_analysis_object

In [ ]:
sample_paths = {
    'std_A_01': 'std_A_01.csv',
    'std_A_02': 'std_A_02.csv',
    'std_B_01': 'std_B_01.csv',
}
sample_to_group = {
    'std_A_01': 'A',
    'std_A_02': 'A',
    'std_B_01': 'B',
}
analysis = create_analysis_object(sample_paths, sample_to_group=sample_to_group, reader='csv').load()

In [ ]:
# Step 1: 降噪 + 指定时间范围可视化
analysis.denoise(method='drift_corrected_moving_average', drift_window=1001, smooth_window=5)
analysis.preview_signal('std_A_01', start_s=0.0, end_s=2.0).head()

In [ ]:
# Step 2: 事件检测（可切换 threshold / zscore_threshold）
analysis.detect_events(detect_method='threshold', detect_params={'sigma_k': 5.0, 'min_duration_s': 2e-4}, baseline_method='rolling_quantile', baseline_params={'window': 501, 'q': 0.5})

In [ ]:
# Step 3: 特征提取（支持自定义特征函数）
def custom_shape_features(seg):
    import numpy as np
    return {'p2p': float(np.max(seg) - np.min(seg)), 'energy': float(np.sum(seg**2))}

feat_df = analysis.extract_features(custom_feature_fns={'shape': custom_shape_features})
feat_df.head()

In [ ]:
# Step 4: 异常事件过滤（noise 标记）
filtered_df = analysis.filter_events(method='isolation_forest', contamination=0.05)
filtered_df[['sample_id', 'event_id', 'quality_tag']].head()

In [ ]:
# Step 5: 10折比较多模型并选择最优
best_pkg = analysis.build_best_model(cv=10, scoring='f1_macro')
best_pkg['best_model'], best_pkg['scores']

In [ ]:
# Step 6: 复用流程 + 最优模型，对新样本逐事件分类
new_paths = {'unknown_01': 'unknown_01.csv'}
pred_df = analysis.classify_new_samples(new_paths, reader='csv')
pred_df[['sample_id', 'event_id', 'start_time_s', 'end_time_s', 'pred_label']].head()